# RuleShift Benchmark

Use this notebook to run the official RuleShift benchmark task inside Kaggle.

| What this notebook does | Why it matters |
| --- | --- |
| Loads the frozen benchmark rows | Ensures every submission is evaluated on the same prompts |
| Normalizes model outputs | Makes scoring robust to a few response formats |
| Scores every episode | Produces the final leaderboard numerator and denominator |
| Registers the Kaggle task | Exposes the benchmark through the expected entry point |

---

> Recommended flow: run the notebook top to bottom once, inspect the smoke-check result, then use the registered task as the official benchmark entry.

## Notebook Map

<div style="background:#111;border:1px solid #2a2a2a;border-radius:6px;padding:18px 22px;font-family:monospace;margin-bottom:8px">
  <div style="display:flex;align-items:center;flex-wrap:wrap;gap:4px">
    <span style="background:#1a1a1a;border:1px solid #2a2a2a;border-radius:4px;padding:4px 10px;color:#aaa;font-size:12px">Paths</span>
    <span style="color:#333;font-size:18px">→</span>
    <span style="background:#1a1a1a;border:1px solid #2a2a2a;border-radius:4px;padding:4px 10px;color:#aaa;font-size:12px">Types</span>
    <span style="color:#444;font-size:18px">╎</span>
    <span style="background:#1a1a1a;border:1px solid #2a2a2a;border-radius:4px;padding:4px 10px;color:#aaa;font-size:12px">Normalize</span>
    <span style="color:#333;font-size:18px">→</span>
    <span style="background:#1a1a1a;border:1px solid #2a2a2a;border-radius:4px;padding:4px 10px;color:#aaa;font-size:12px">Score</span>
    <span style="color:#444;font-size:18px">╎</span>
    <span style="background:#1a1a1a;border:1px solid #2a2a2a;border-radius:4px;padding:4px 10px;color:#aaa;font-size:12px">Load Rows</span>
    <span style="color:#444;font-size:18px">╎</span>
    <span style="background:#1a1a1a;border:1px solid #2a2a2a;border-radius:4px;padding:4px 10px;color:#aaa;font-size:12px">Register Task</span>
    <span style="color:#444;font-size:18px">╎</span>
    <span style="background:#1a1a1a;border:1px solid #2a2a2a;border-radius:4px;padding:4px 10px;color:#aaa;font-size:12px">Smoke Check</span>
    <span style="color:#333;font-size:18px">→</span>
    <span style="background:#1a1a1a;border:1px solid #2a2a2a;border-radius:4px;padding:4px 10px;color:#ccc;font-size:12px">Choose Task</span>
  </div>
</div>

## Phase 1 — Runtime setup

Defines the constants, types, and filesystem paths that every later cell depends on.

### Step 1 — Locate the packaged benchmark data

This cell defines the runtime dataset paths before any benchmark logic starts.

- **Kaggle runtime root** — canonical source of truth for the packaged dataset
- **`data/public_leaderboard_rows.json`** — public benchmark rows loaded by default
- **`RULESHIFT_PRIVATE_DATASET_ROOT`** — optional private benchmark rows location

The notebook assumes the Kaggle runtime dataset is mounted at the canonical root and resolves the row file paths once here for later cells to reuse.

In [ ]:
import os
from pathlib import Path

import kaggle_benchmarks as kbench

PRIVATE_DATASET_ROOT_ENV_VAR = "RULESHIFT_PRIVATE_DATASET_ROOT"

RUNTIME_ROOT = Path("/kaggle/input/datasets/raptorengineer/ruleshift-runtime")
PUBLIC_ROWS_PATH = RUNTIME_ROOT / "public_leaderboard_rows.json"

PRIVATE_DATASET_ROOT = os.environ.get(PRIVATE_DATASET_ROOT_ENV_VAR)
PRIVATE_ROWS_PATH = (
    Path(PRIVATE_DATASET_ROOT) / "private_leaderboard_rows.json"
    if PRIVATE_DATASET_ROOT
    else None
)

### Step 2 — Define shared constants and types

The runtime is intentionally split into smaller sections so each concern can be understood on its own.

| Runtime section | What it contains |
| --- | --- |
| Shared constants and types | Labels, response schema, path config, and benchmark-wide settings |
| Response normalization | Logic that converts different model outputs into the expected labels |
| Scoring helpers | Functions that compare predictions against the frozen targets |
| Dataset loading | Functions that read the public and private benchmark rows |

This first code block defines the shared vocabulary that every later cell depends on.

In [ ]:
from dataclasses import dataclass
from enum import Enum
from typing import Final

class Label(str, Enum):
    type_a = "type_a"
    type_b = "type_b"

PROBE_COUNT: Final[int] = 4

_ALLOWED_LABELS: Final[frozenset[str]] = frozenset(label.value for label in Label)
_PROBE_FIELDS: Final[tuple[str, ...]] = ("probe_6", "probe_7", "probe_8", "probe_9")

@dataclass(frozen=True)
class BinaryResponse:
    probe_6: Label
    probe_7: Label
    probe_8: Label
    probe_9: Label

    def as_tuple(self) -> tuple[str, str, str, str]:
        """Returns the four probe labels as normalized strings.

        Converts each stored probe label into its canonical lowercase string.

        Returns:
            A four-item tuple containing the normalized probe labels.
        """
        return (
            _coerce_label(self.probe_6, "probe_6"),
            _coerce_label(self.probe_7, "probe_7"),
            _coerce_label(self.probe_8, "probe_8"),
            _coerce_label(self.probe_9, "probe_9"),
        )

class KaggleExecutionError(RuntimeError):
    pass

## Phase 2 — Scoring logic

Implements response normalization and the episode-level scoring functions.

### Step 3 — Normalize model responses

Model outputs are not always shaped the same way, so this section standardizes them before scoring.

| Accepted shape | Example |
| --- | --- |
| Structured response object | `BinaryResponse(...)` |
| Dictionary-like response | `{"probe_6": "type_a", ...}` |
| Plain text response | `type_a, type_b, type_a, type_b` |

The helpers are forgiving about format, but strict about labels: each normalized answer must end up as exactly four values drawn from `type_a` and `type_b`.

In [ ]:
def normalize_binary_response(response: object) -> tuple[str, ...] | None:
    """Normalizes a model response into a canonical four-label tuple.

    Accepts structured objects, mapping-like objects, or plain text responses.

    Args:
        response: The raw model response to normalize.

    Returns:
        A normalized four-label tuple, or `None` when the response cannot be normalized.
    """
    if response is None:
        return None
    if isinstance(response, BinaryResponse):
        return response.as_tuple()
    if isinstance(response, str):
        return _parse_text_response(response)
    br = _try_coerce(response)
    return br.as_tuple() if br is not None else None

def _parse_text_response(text: str) -> tuple[str, ...] | None:
    """Parses a plain-text response into normalized probe labels.

    Splits text on commas after trimming whitespace, backticks, and line breaks.

    Args:
        text: The plain-text response to parse.

    Returns:
        A normalized four-label tuple, or `None` if parsing does not yield four valid labels.
    """
    tokens = tuple(
        t.strip().lower()
        for t in text.strip().strip("`").replace("\n", ",").split(",")
        if t.strip()
    )
    return _norm_labels(tokens)

def _try_coerce(response: object) -> BinaryResponse | None:
    """Attempts to coerce a response object into `BinaryResponse`.

    Supports dictionaries, mapping-like objects, and objects exposing probe attributes.

    Args:
        response: The response object to coerce.

    Returns:
        A `BinaryResponse` instance, or `None` when coercion fails.
    """
    if isinstance(response, dict):
        vals = response
    elif hasattr(response, "__getitem__") and hasattr(response, "keys"):
        try:
            vals = {f: response[f] for f in _PROBE_FIELDS}
        except (KeyError, TypeError):
            return None
    elif all(hasattr(response, f) for f in _PROBE_FIELDS):
        vals = {f: getattr(response, f) for f in _PROBE_FIELDS}
    else:
        return None
    try:
        labels = tuple(Label(_coerce_label(vals[f], f)) for f in _PROBE_FIELDS)
    except (KeyError, TypeError):
        return None
    return BinaryResponse(*labels)

def _coerce_label(value: object, field: str) -> str:
    """Normalizes a single label value with field-aware errors.

    Wraps label normalization so invalid values report which field failed.

    Args:
        value: The label value to normalize.
        field: The field name associated with the value.

    Returns:
        The normalized label string.
    """
    try:
        return _normalize_label(value)
    except ValueError as exc:
        raise ValueError(f"invalid field {field}: {value!r}") from exc

def _norm_labels(
    labels: tuple[str, ...] | tuple[Label, ...] | None,
) -> tuple[str, ...] | None:
    """Normalizes a sequence of labels and enforces the probe count.

    Validates that the sequence contains exactly four supported labels.

    Args:
        labels: The label sequence to normalize.

    Returns:
        A normalized four-label tuple, or `None` if validation fails.
    """
    if labels is None:
        return None
    try:
        result = tuple(
            _normalize_label(lbl.value if isinstance(lbl, Label) else lbl)
            for lbl in labels
        )
    except ValueError:
        return None
    return result if len(result) == PROBE_COUNT else None

def _normalize_label(value: object) -> str:
    """Converts a supported label value to its canonical string form.

    Accepts enum values and other value-like objects with a `.value` attribute.

    Args:
        value: The value to normalize.

    Returns:
        The lowercase canonical label string.
    """
    if isinstance(value, Enum):
        value = value.value
    elif hasattr(value, "value"):
        value = value.value
    normalized = str(value).strip().lower()
    if normalized not in _ALLOWED_LABELS:
        raise ValueError(f"unknown label: {value}")
    return normalized

### Step 4 — Score each benchmark episode

This section converts one benchmark prompt into one measurable score.

<div style="background:#111;border:1px solid #2a2a2a;border-radius:6px;padding:18px 22px;font-family:monospace;margin-bottom:8px">
  <div style="display:flex;align-items:center;flex-wrap:wrap;gap:4px">
    <span style="background:#1a1a1a;border:1px solid #2a2a2a;border-radius:4px;padding:4px 10px;color:#aaa;font-size:12px">Prompt the model</span>
    <span style="color:#333;font-size:18px">→</span>
    <span style="background:#1a1a1a;border:1px solid #2a2a2a;border-radius:4px;padding:4px 10px;color:#aaa;font-size:12px">Normalize the response</span>
    <span style="color:#333;font-size:18px">→</span>
    <span style="background:#1a1a1a;border:1px solid #2a2a2a;border-radius:4px;padding:4px 10px;color:#aaa;font-size:12px">Compare with targets</span>
    <span style="color:#333;font-size:18px">→</span>
    <span style="background:#1a1a1a;border:1px solid #2a2a2a;border-radius:4px;padding:4px 10px;color:#ccc;font-size:12px">Return (numerator, denominator)</span>
  </div>
</div>

If the model call fails or returns something unscoreable, the runtime raises an explicit error so the failure is easy to diagnose.

In [ ]:
def score_episode(
    predictions: tuple[str, ...] | tuple[Label, ...] | None,
    targets: tuple[str, ...] | tuple[Label, ...],
) -> tuple[int, int]:
    """Scores one episode by comparing predictions with target labels.

    Treats invalid or missing predictions as zero correct answers.

    Args:
        predictions: The predicted probe labels.
        targets: The target probe labels.

    Returns:
        A `(numerator, denominator)` tuple for the episode score.
    """
    norm_targets = _norm_labels(targets)
    if norm_targets is None:
        raise ValueError(f"targets must contain exactly {PROBE_COUNT} valid labels")
    norm_preds = _norm_labels(predictions)
    if norm_preds is None:
        return (0, PROBE_COUNT)
    return (
        sum(p == t for p, t in zip(norm_preds, norm_targets)),
        PROBE_COUNT,
    )

def run_binary_task(
    *,
    llm: object,
    prompt_binary: str,
    probe_targets: tuple[str, ...] | tuple[Label, ...],
) -> tuple[int, int]:
    """Runs one benchmark episode against the provided model.

    Prompts the model, normalizes its response, and scores the resulting labels.

    Args:
        llm: The model interface used to produce a response.
        prompt_binary: The benchmark prompt to send to the model.
        probe_targets: The expected probe labels for scoring.

    Returns:
        A `(numerator, denominator)` tuple for the episode score.
    """
    try:
        response = llm.prompt(prompt_binary, schema=BinaryResponse)
    except Exception as exc:
        raise KaggleExecutionError("llm.prompt failed") from exc

    try:
        normalized = normalize_binary_response(response)
    except ValueError as exc:
        raise KaggleExecutionError(f"invalid binary response: {exc}") from exc

    if normalized is None:
        raise KaggleExecutionError(
            f"unscoreable response of type {type(response).__name__}"
        )
    return score_episode(normalized, probe_targets)

## Phase 3 — Dataset loading

Materializes the frozen evaluation rows from the packaged runtime dataset.

### Step 5 — Load the frozen benchmark rows

This part materializes the full evaluation set that the task will iterate over.

| Split | Source | When it is used |
| --- | --- | --- |
| Public rows | Packaged notebook dataset | Always |
| Private rows | `RULESHIFT_PRIVATE_DATASET_ROOT` | Only when the private dataset is available |

At the end of the cell, `leaderboard_rows` becomes the single source of truth for the benchmark loop. If no rows are available, the notebook stops immediately.

In [ ]:
import json

def load_public_rows() -> list[dict[str, object]]:
    """Loads the packaged public leaderboard rows.

    Reads the public benchmark rows from the runtime dataset.

    Returns:
        A list of normalized public benchmark rows.
    """
    return _load_rows(PUBLIC_ROWS_PATH)

def load_private_rows(
    private_rows_path: Path | str | None = None,
) -> list[dict[str, object]]:
    """Loads private leaderboard rows from the configured dataset location.

    Falls back to the environment-configured private dataset path when none is provided.

    Args:
        private_rows_path: An optional explicit path to the private rows file.

    Returns:
        A list of normalized private benchmark rows.
    """
    if private_rows_path is None:
        if PRIVATE_ROWS_PATH is None:
            raise FileNotFoundError(
                f"Private dataset not found. Set {PRIVATE_DATASET_ROOT_ENV_VAR} or "
                "pass private_rows_path explicitly."
            )
        private_rows_path = PRIVATE_ROWS_PATH
    return _load_rows(Path(private_rows_path))

def _load_rows(path: Path) -> list[dict[str, object]]:
    """Loads benchmark rows from disk and normalizes their probe targets.

    Extracts the fields required by the runtime from the raw JSON payload.

    Args:
        path: The path to the JSON file containing benchmark rows.

    Returns:
        A list of normalized benchmark row dictionaries.
    """
    rows = json.loads(path.read_text("utf-8"))
    return [
        {
            "episode_id": row["episode_id"],
            "split": row["split"],
            "prompt_binary": row["prompt_binary"],
            "probe_targets": _normalize_probe_targets(row["probe_targets"]),
        }
        for row in rows
    ]

def _normalize_probe_targets(values: object) -> tuple[str, ...]:
    """Validates and normalizes stored probe target labels.

    Ensures the targets are provided as a sequence of exactly four valid labels.

    Args:
        values: The raw probe target values to validate.

    Returns:
        A normalized four-label tuple.
    """
    if not isinstance(values, list | tuple):
        raise ValueError("probe_targets must be a list or tuple")
    labels = _norm_labels(tuple(values))
    if labels is None:
        raise ValueError(f"probe_targets must contain exactly {PROBE_COUNT} valid labels")
    return labels

leaderboard_rows = load_public_rows()
if PRIVATE_ROWS_PATH is not None:
    leaderboard_rows.extend(load_private_rows())

if not leaderboard_rows:
    raise RuntimeError("No benchmark episodes were loaded.")

## Phase 4 — Task registration

Wraps the scoring loop into the `@kbench.task` entry point exposed to Kaggle.

### Step 6 — Register the official Kaggle task

This cell turns the scoring loop into a Kaggle Benchmarks task.

Execution sequence:

1. Iterate through every row in `leaderboard_rows`.
2. Send the prompt to the provided `llm`.
3. Score the four probe predictions for that episode.
4. Aggregate the episode totals into a final benchmark result.
5. Save a compact summary so the notebook can display it after the run.

This is the main evaluation cell: everything above prepares data and helpers for this loop.

In [ ]:
_RULESHIFT_RESULT = None


@kbench.task(
    name="ruleshift_benchmark_v1_binary",
    description=(
        "Cognitive flexibility benchmark: infer a hidden rule shift from "
        "sparse contradictory evidence in a sequence of marker records, "
        "then predict four post-shift probe outputs."
    ),
)
def ruleshift_benchmark_v1_binary(llm) -> tuple[int, int]:
    """Evaluates the benchmark across all loaded rows.

    Aggregates episode-level scores and stores a readable summary in `_RULESHIFT_RESULT`.

    Args:
        llm: The Kaggle Benchmarks-compatible model interface to evaluate.

    Returns:
        A `(numerator, denominator)` tuple for the full benchmark score.
    """
    global _RULESHIFT_RESULT

    numerator = 0
    denominator = 0
    for row in leaderboard_rows:
        num_correct, total = run_binary_task(
            llm=llm,
            prompt_binary=row["prompt_binary"],
            probe_targets=row["probe_targets"],
        )
        numerator += int(num_correct)
        denominator += int(total)

    if denominator == 0:
        raise RuntimeError("evaluation produced a zero denominator")

    _RULESHIFT_RESULT = {
        "score": numerator / denominator,
        "numerator": numerator,
        "denominator": denominator,
        "episodes": len(leaderboard_rows),
    }

    print(json.dumps(_RULESHIFT_RESULT, indent=2))
    return (numerator, denominator)

## Phase 5 — Validation and selection

Runs the task end to end inside the notebook and binds the entry point for Kaggle.

### Step 7 — Run a notebook smoke check

This cell executes the registered task once inside the notebook and displays the resulting summary.

- **Sanity check** — confirms the notebook can run end to end
- **Quick inspection** — shows score, numerator, denominator, and episode count
- **Debugging help** — surfaces runtime issues before submission

If this smoke check succeeds, the notebook is in a good state for Kaggle task selection.

> **Expected output:** `_RULESHIFT_RESULT` populated with `score`, `numerator`, `denominator`, and `episodes`.

In [ ]:
ruleshift_benchmark_v1_binary(kbench.llm)
if _RULESHIFT_RESULT is None:
    raise RuntimeError("ruleshift_benchmark_v1_binary did not populate _RULESHIFT_RESULT")

_RULESHIFT_RESULT

### Step 8 — Mark the official entry point

The final step is to tell Kaggle which registered task name should be used as the notebook entry point.

In [ ]:
%choose ruleshift_benchmark_v1_binary